In [11]:
import numpy as np
import sys
import types
import os
# =========================================================
# 0. HARD DISABLE MPI (ROBUST VERSION)
# =========================================================
# import os
# import sys
# import types

# os.environ["OMP_NUM_THREADS"] = "1"
# os.environ["DESI_DISABLE_MPI"] = "1"


# # -------------------------
# # Fake MPI implementation
# # -------------------------
# class FakeComm:
#     def Get_rank(self): return 0
#     def Get_size(self): return 1
#     def Barrier(self): pass
#     def bcast(self, x, root=0): return x
#     def allreduce(self, x, op=None): return x
#     def gather(self, x, root=0): return [x]
#     def allgather(self, x): return [x]


# class FakeStatus:
#     pass


# class FakeMPI:
#     COMM_WORLD = FakeComm()
#     COMM_SELF = FakeComm()

#     ANY_SOURCE = -1
#     ANY_TAG = -1

#     SUM = None
#     MAX = None
#     MIN = None

#     Status = FakeStatus

#     @staticmethod
#     def Get_processor_name():
#         return "fake-node"


# # -------------------------
# # Fake mpi4py module tree
# # -------------------------
# mpi4py = types.ModuleType("mpi4py")
# mpi4py.MPI = FakeMPI

# sys.modules["mpi4py"] = mpi4py
# sys.modules["mpi4py.MPI"] = FakeMPI



# ---------------------------
# NOW safe to import desilike
# ---------------------------
from desilike.theories.galaxy_clustering import DampedBAOWigglesTracerPowerSpectrumMultipoles
from desilike.observables.galaxy_clustering import TracerPowerSpectrumMultipolesObservable
from desilike.likelihoods import ObservablesGaussianLikelihood
from desilike.profilers import MinuitProfiler


# ---------------------------
# Load data
# ---------------------------
filename = "../data/monopoles/box/mag=-21.2_sep=1.0-150.0_binsep=2.0_full.npz"

data = np.load(filename)
print("keys:", data.files)

s = data["s"]
xi = data["xi0"]


# ---------------------------
# Fake covariance (temporary)
# ---------------------------
sigma_xi = 0.05 * np.abs(xi)
cov = np.diag(sigma_xi**2)


# ---------------------------
# Theory
# ---------------------------
theory = TracerPowerSpectrumMultipolesObservable()

#theory.init.update(
#    k=s
#)

params = theory.params
params['alpha_iso'].update(value=1.0, prior={'limits': [0.8, 1.2]})
params['sigma_nl'].update(value=5.0, prior={'limits': [0., 20.]})

# broadband
for name in params.select(basename='a*'):
    params[name].update(fixed=False)


# ---------------------------
# Likelihood
# ---------------------------
observable = TracerPowerSpectrumMultipolesObservable(data=xi, covariance=cov, theory=theory)
likelihood = ObservablesGaussianLikelihood(observables=[observable])


# ---------------------------
# Fit
# ---------------------------
profiler = MinuitProfiler(likelihood)
bestfit = profiler.maximize()


print("\nalpha =", bestfit.params['alpha'])
print("Sigma_nl =", bestfit.params['Sigma_nl'])
print("chi2 =", bestfit.chi2)

keys: ['s', 'xi0']


KeyError: 'Parameter alpha_iso not found'

In [12]:
theory.all_params

PipelineError: Error in method initialize of <desilike.observables.galaxy_clustering.power_spectrum.TracerPowerSpectrumMultipolesObservable object at 0x79b5e9674f90>

### Second Try

In [5]:
import numpy as np

# ---------------------------
# desilike imports
# ---------------------------
from desilike.theories.galaxy_clustering import (
    FlexibleBAOWigglesTracerCorrelationFunctionMultipoles,
    BAOPowerSpectrumTemplate
)
from desilike.observables.galaxy_clustering import (
    TracerCorrelationFunctionMultipolesObservable
)
from desilike.likelihoods import ObservablesGaussianLikelihood
from desilike.profilers import MinuitProfiler


# =========================================================
# 1. Load your data
# =========================================================
filename = "../data/monopoles/box/mag=-21.2_sep=1.0-150.0_binsep=2.0_full.npz"

data = np.load(filename)
print("keys:", data.files)

s = data["s"]
xi = data["xi0"]


# =========================================================
# 2. Covariance (temporary)
# =========================================================
sigma_xi = 0.05 * np.abs(xi)
cov = np.diag(sigma_xi**2)


# =========================================================
# 3. BAO THEORY (IMPORTANT CHANGE)
# =========================================================
z = 0.5  # <-- set appropriately for your sample

template = BAOPowerSpectrumTemplate(
    z=z,
    fiducial='Planck2018FullFlatLCDM',
    apmode='qiso'   # gives qiso = alpha
)

theory = FlexibleBAOWigglesTracerCorrelationFunctionMultipoles(
    template=template,
    broadband='pcs2',   # standard choice
    wiggles='pcs'
)


# =========================================================
# 4. OBSERVABLE (CRITICAL FIX)
# =========================================================
# desilike expects structured data, not raw array

observable = TracerCorrelationFunctionMultipolesObservable(
    data={'xi': {0: xi}},   # monopole ℓ=0
    covariance=cov,
    slim={0: [50., 150., s[1] - s[0]]},
    theory=theory
)

# Freeze data vector (important!)
observable.init.update(data=observable.flatdata)


# =========================================================
# 5. PARAMETERS
# =========================================================
params = theory.params

# BAO scale (THIS is alpha)
params['qiso'].update(value=1.0, prior={'limits': [0.8, 1.2]})

# Bias (optional but recommended)
if 'b1' in params.names():
    params['b1'].update(value=1.5)

# Let broadband float
for name in params.select(basename='a*'):
    params[name].update(fixed=False)

for name in params.select(basename='m*'):
    params[name].update(fixed=False)


# =========================================================
# 6. LIKELIHOOD
# =========================================================
likelihood = ObservablesGaussianLikelihood(observables=[observable])


# =========================================================
# 7. FIT
# =========================================================
profiler = MinuitProfiler(likelihood)

bestfit = profiler.maximize()


# =========================================================
# 8. RESULTS
# =========================================================
print("\n===== BEST FIT =====")

print("qiso (alpha) =", bestfit.params['qiso'])

if 'b1' in bestfit.params:
    print("b1 =", bestfit.params['b1'])

print("chi2 =", bestfit.chi2)

keys: ['s', 'xi0']


PipelineError: Error in method initialize of <desilike.observables.galaxy_clustering.correlation_function.TracerCorrelationFunctionMultipolesObservable object at 0x772f27a1a7d0>

In [19]:
theory.params


ParameterCollection(['b1', 'dbeta', 'sigmas', 'sigmapar', 'sigmaper', 'al0_-2', 'al0_-1', 'al0_0', 'al0_1', 'al0_2', 'al2_-2', 'al2_-1', 'al2_0', 'al2_1', 'al2_2', 'al4_-2', 'al4_-1', 'al4_0', 'al4_1', 'al4_2'])

In [4]:
import cosmoprimo.fiducial as fid

print([name for name in dir(fid) if not name.startswith("_")])

['AbacusSummit', 'AbacusSummitBase', 'AbacusSummit_params', 'BOSS', 'Cosmology', 'DESI', 'Planck2018FullFlatLCDM', 'TabulatedDESI', 'Uchuu', 'constants', 'get_engine', 'np', 'os', 'save_TabulatedDESI']
